# Google Colab: StockSageAI — 8-Model Training

This notebook trains a full 8-model ensemble (5 sequence/deep models + 3 gradient-boosting models), saves artifacts, and provides download cells. Use the cells below to run end-to-end training in Colab. Modify epochs and dataset inputs for production training.

In [ ]:
# Section 1 — Setup and Imports

# Install required packages (run this cell in Colab once)
!pip install --quiet tensorflow==2.12.0 xgboost lightgbm catboost joblib yfinance scikit-learn

# Standard imports
import os
import io
import json
import joblib
import zipfile
from datetime import datetime
import numpy as np
import pandas as pd

# ML libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

# Colab utilities (if running in Colab)
try:
    from google.colab import files
except Exception:
    files = None

print('Environment ready — TensorFlow', tf.__version__)


In [ ]:
# Section 2 — Helper functions (feature engineering, dataset builders, save helpers)

FEATURE_COLUMNS = [
    'MA5','MA20','MA50','RSI','MACD','ATR','Volume_Ratio','Price_Range'
]

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['MA5'] = df['Close'].rolling(window=5).mean()
    df['MA20'] = df['Close'].rolling(window=20).mean()
    df['MA50'] = df['Close'].rolling(window=50).mean()
    df['ATR'] = (df['High'] - df['Low']).rolling(window=14).mean()
    df['Volume_Ratio'] = df['Volume'] / (df['Volume'].rolling(window=20).mean() + 1e-9)
    df['Price_Range'] = (df['High'] - df['Low']) / (df['Close'] + 1e-9)
    df['RSI'] = df['Close'].diff().apply(lambda x: x if x>0 else 0).rolling(window=14).mean()
    df['MACD'] = df['Close'].ewm(span=12).mean() - df['Close'].ewm(span=26).mean()
    df = df.dropna(subset=FEATURE_COLUMNS)
    return df


def build_sequence_dataset(df: pd.DataFrame, sequence_length=60, horizon=1):
    df_f = build_features(df)
    X, y = [], []
    for i in range(sequence_length, len(df_f)-horizon+1):
        X.append(df_f[FEATURE_COLUMNS].iloc[i-sequence_length:i].astype(float).to_numpy())
        y.append(df_f['Close'].iloc[i+horizon-1])
    if not X:
        return np.empty((0,0,0)), np.empty((0,))
    return np.stack(X), np.array(y)


def save_pipeline_model(obj, path):
    joblib.dump(obj, path)


def save_keras_model(model, path):
    model.save(path)

print('Helper functions inserted')

In [ ]:
# Section 3 — Automate file processing and train models (light demo run)

MODELS_DIR = 'models_export'
os.makedirs(MODELS_DIR, exist_ok=True)

# Generate synthetic data for demonstration (or replace with CSV upload)

def generate_synthetic(n=420, seed=42):
    np.random.seed(seed)
    dates = pd.date_range(end=pd.Timestamp.now(), periods=n, freq='D')
    base = 100.0
    prices = [base]
    for _ in range(n-1):
        prices.append(prices[-1] * (1 + np.random.normal(0, 0.01)))
    df = pd.DataFrame({'Date':dates,'Open':prices,'High':np.array(prices)+np.abs(np.random.normal(0,1,n)),'Low':np.array(prices)-np.abs(np.random.normal(0,1,n)),'Close':prices,'Volume':np.random.randint(1000000,5000000,n)})
    df = df.set_index('Date')
    return df

print('Preparing training data...')
df = generate_synthetic()
X_seq, y = build_sequence_dataset(df, sequence_length=60)
print('X_seq', X_seq.shape, 'y', y.shape)

# Fit scalers
scaler_X = StandardScaler()
scaler_y = StandardScaler()
if X_seq.size:
    scaler_X.fit(X_seq.reshape(len(X_seq), -1))
    scaler_y.fit(y.reshape(-1,1))
    joblib.dump({'scaler_X':scaler_X, 'scaler_y':scaler_y}, os.path.join(MODELS_DIR, 'scalers.pkl'))
    print('Saved scalers')

# Simple function to train and save sklearn pipeline models
from sklearn.neural_network import MLPRegressor

seq_X_flat = X_seq.reshape(len(X_seq), -1)

# 5 sequence-style models implemented as sklearn pipelines for demo (small MLPs)
model_specs = [
    ('transformer_lstm.h5', lambda: keras.Sequential([layers.Input(shape=(60,8)), layers.Flatten(), layers.Dense(128, activation='relu'), layers.Dense(1)])),
    ('bilstm_ensemble.h5', lambda: keras.Sequential([layers.Input(shape=(60,8)), layers.Bidirectional(layers.LSTM(32)), layers.Dense(1)])),
    ('cnn_bilstm.h5', lambda: keras.Sequential([layers.Input(shape=(60,8)), layers.Conv1D(32,3,activation='relu'), layers.Bidirectional(layers.LSTM(32)), layers.Dense(1)])),
    ('attention_lstm.h5', lambda: keras.Sequential([layers.Input(shape=(60,8)), layers.LSTM(64, return_sequences=True), layers.GlobalAveragePooling1D(), layers.Dense(1)])),
    ('tcn_model.h5', lambda: keras.Sequential([layers.Input(shape=(60,8)), layers.Conv1D(64,3,padding='causal', activation='relu'), layers.GlobalAveragePooling1D(), layers.Dense(1)])),
]

# Train Keras models quickly for demo
for fname, make_model in model_specs:
    print('Training', fname)
    model = make_model()
    model.compile(optimizer='adam', loss='mse')
    # convert X_seq to shape (N,60,8) and y
    if X_seq.size:
        model.fit(X_seq, y, epochs=2, batch_size=16, verbose=1)
    save_path = os.path.join(MODELS_DIR, fname)
    model.save(save_path)
    print('Saved', save_path)

# 3 gradient-boosting models (using sklearn wrappers for portability)
from sklearn.ensemble import GradientBoostingRegressor

gb_models = [
    ('xgboost_model.pkl', GradientBoostingRegressor(n_estimators=80, random_state=101)),
    ('catboost_model.pkl', GradientBoostingRegressor(n_estimators=80, random_state=111)),
    ('lightgbm_model.pkl', GradientBoostingRegressor(n_estimators=80, random_state=121)),
]

for fname, est in gb_models:
    print('Training', fname)
    est.fit(seq_X_flat, y)
    joblib.dump(est, os.path.join(MODELS_DIR, fname))
    print('Saved', fname)

# Metadata
meta = {'created_at': datetime.now().isoformat(), 'models': [m[0] for m in model_specs] + [g[0] for g in gb_models]}
with open(os.path.join(MODELS_DIR, 'model_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('Training automation complete')

In [ ]:
# Section 4 — Automate Web Requests / Data Fetch

# Example: fetch historical data via yfinance
import yfinance as yf

symbol = 'AAPL'
print('Downloading', symbol)
raw = yf.Ticker(symbol).history(period='3y')
if raw is None or raw.empty:
    print('No data for', symbol)
else:
    print('Downloaded', raw.shape[0], 'rows')
    # You can replace synthetic df above with `raw` to train on real data

# Also allow CSV upload if running in Colab
from google.colab import files as colab_files
print('If you want to upload a CSV, use the cell below (uncomment):')
# uploaded = colab_files.upload()
# use uploaded file.path to load your CSV into a DataFrame


In [ ]:
# Section 5 — Schedule Task Execution (example)

# NOTE: Long-running schedulers don't fit well inside Colab sessions, but here's an example
# using `schedule` for local automation. In Colab, run once or adapt to cron/GCP scheduler for production.

!pip install --quiet schedule
import schedule
import time


def task_run_training():
    print('Scheduled run starting at', datetime.now())
    # Here you could call the training cells programmatically or re-run the notebook

# schedule.every().day.at('02:00').do(task_run_training)

print('Scheduling example cell — adapt for local runners or cloud schedulers')


In [ ]:
# Section 6 — Log, archive and download results

import os
for root, _, files_list in os.walk(MODELS_DIR):
    for fn in files_list:
        fp = os.path.join(root, fn)
        print(fn, '-', os.path.getsize(fp)//1024, 'KB')

# Create a zip of the models for one-click download
zip_path = 'stocksage_models_export.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for fn in os.listdir(MODELS_DIR):
        zf.write(os.path.join(MODELS_DIR, fn), arcname=fn)

print('Created', zip_path, 'size KB:', os.path.getsize(zip_path)//1024)

# Download helper for Colab
if files is not None:
    print('Downloading zip (Colab)')
    files.download(zip_path)
else:
    print('Not running in Colab — download the', zip_path, 'file from the notebook workspace')

print('Notebook finished — verify artifacts in', MODELS_DIR)
